# Local Spark retail star schema benchmark example

Generate repeatable synthetic data and run SQL templates in a local Spark session. Results depend on local hardware and Spark configuration.

In [ ]:
%pip install "great-generator[spark]"

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "benchmarks/datasets/retail_star_schema/generate_dataset.py", "--output", "output/retail_star_schema", "--format", "parquet", "--overwrite"])

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("great-generator-retail-benchmark").getOrCreate()
base = "output/retail_star_schema"
for table in ["dim_customer", "dim_product", "dim_store", "dim_date", "fact_sales"]:
    spark.read.parquet(f"{base}/{table}").createOrReplaceTempView(table)
spark.sql("SELECT business_date, COUNT(*) AS sales_rows, SUM(sales_amount) AS sales_amount FROM fact_sales GROUP BY business_date ORDER BY business_date").show()